# Name That LLM — Prediction bundle submission

Runs the deterministic prediction bundle selected from saved V0 validation predictions. The selected six-model equal-probability blend scored 60.0% on that validation set. Attach the private prediction bundle, the public SmolLM3 and Qwen3 base datasets, and the Ministral 3 Kaggle Model through kernel metadata before running this notebook. The bundle omits public base weights to keep upload size manageable; this notebook mounts them locally for offline inference.

The validation score is a model-selection result and may be optimistic because all candidate blends were compared on the same validation rows.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, zipfile

BUNDLE_DATASET_DIR = Path('/kaggle/input/nlm-prediction-bundle')
BUNDLE_ROOT = BUNDLE_DATASET_DIR / 'prediction_v0_49eb0554416f_bundle_external_bases'
BUNDLE_ZIP = BUNDLE_DATASET_DIR / 'prediction_v0_49eb0554416f_bundle_external_bases.zip'
BUNDLE_DIR = Path('/kaggle/working/prediction_bundle')
TEST_CANDIDATES = [Path('/kaggle/input/competitions/name-that-llm/test.csv'), Path('/kaggle/input/name-that-llm/test.csv')]
OUTPUT = Path('/kaggle/working/submission.parquet')
EXPECTED_EXPERIMENT_ID = 'prediction_v0_49eb0554416f'
EXPECTED_VALIDATION_ACCURACY = 0.6

if (BUNDLE_ROOT / 'prediction_bundle.json').is_file():
    BUNDLE_DIR = BUNDLE_ROOT
elif BUNDLE_ZIP.is_file():
    BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(BUNDLE_ZIP) as archive:
        archive.extractall(BUNDLE_DIR)
else:
    raise FileNotFoundError(f'Bundle folder or ZIP not found under {BUNDLE_DATASET_DIR}')
assert (BUNDLE_DIR / 'prediction_bundle.json').is_file(), 'Prediction bundle manifest is missing'
selection = json.loads((BUNDLE_DIR / 'prediction_bundle.json').read_text())
assert selection['experiment_id'] == EXPECTED_EXPERIMENT_ID, selection['experiment_id']
assert abs(selection['validation_accuracy'] - EXPECTED_VALIDATION_ACCURACY) < 1e-12, selection['validation_accuracy']
TEST_CSV = next((path for path in TEST_CANDIDATES if path.is_file()), None)
assert TEST_CSV is not None, f'Competition test.csv not found in: {TEST_CANDIDATES}'
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
sys.path.insert(0, str(BUNDLE_DIR))
# Kaggle's image currently includes torchao 0.10.0, incompatible with the
# installed PEFT release. This inference path uses no torchao features.
try:
    from importlib.metadata import PackageNotFoundError, version
    from packaging.version import Version
    try:
        if Version(version('torchao')) <= Version('0.16.0'):
            subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'],
                           check=True, capture_output=True, text=True)
    except PackageNotFoundError:
        pass
except ImportError:
    pass
import torch
assert torch.cuda.is_available(), 'Kaggle did not provide a CUDA GPU; aborting before inference.'
from src.inference import predict_prediction_bundle

# Resolve the public, versioned Kaggle base-model inputs. SmolLM3 is distributed
# as shards without an HF index, so create the index from each shard's keys.
SMOL_BASE_DATA = Path('/kaggle/input/smollm3-3b-base-36l-text')
QWEN_BASE = Path('/kaggle/input/qwen3-0-6b/Qwen3-0.6B')
MINISTRAL_BASE = Path('/kaggle/input/models/mistral-ai/ministral-3/transformers/ministral-3-3b-base-2512/1')
SMOL_BASE = Path('/kaggle/working/base_models/smollm3_3b')
SMOL_BASE.mkdir(parents=True, exist_ok=True)
for item in SMOL_BASE_DATA.iterdir():
    target = SMOL_BASE / item.name
    if target.exists():
        continue
    if item.name == 'config.json':
        shutil.copy2(item, target)
    else:
        target.symlink_to(item)
# Some Kaggle mirrors use a temporary model_type alias. SmolLM3 is registered
# by Transformers as `smollm3`; normalize the local config for that architecture.
smol_config_path = SMOL_BASE / 'config.json'
smol_config = json.loads(smol_config_path.read_text())
if smol_config.get('model_type') == 'smollm3_text_base':
    smol_config['model_type'] = 'smollm3'
    smol_config_path.write_text(json.dumps(smol_config))
from transformers.models.auto.configuration_auto import CONFIG_MAPPING
assert 'smollm3' in CONFIG_MAPPING, 'This Kaggle Transformers build does not support SmolLM3'
smol_shards = sorted(SMOL_BASE.glob('*.safetensors'))
assert smol_shards, f'SmolLM3 weights not found under {SMOL_BASE_DATA}'
smol_index = SMOL_BASE / 'model.safetensors.index.json'
if not smol_index.exists():
    from safetensors import safe_open
    weight_map = {}
    for shard in smol_shards:
        with safe_open(str(shard), framework='pt', device='cpu') as sf:
            weight_map.update({key: shard.name for key in sf.keys()})
    smol_index.write_text(json.dumps({'metadata': {}, 'weight_map': weight_map}))
assert (QWEN_BASE / 'config.json').is_file(), f'Qwen base not found: {QWEN_BASE}'
assert (MINISTRAL_BASE / 'config.json').is_file(), f'Ministral base not found: {MINISTRAL_BASE}'
BASE_OVERRIDES = {
    'smollm3_3b': str(SMOL_BASE),
    'qwen06': str(QWEN_BASE),
    'ministral3_3b': str(MINISTRAL_BASE),
}

# Hidden test rows can drive larger attention allocations than the public sample.
# Start at batch 4 and retry smaller batches only if CUDA reports OOM.
import src.inference as prediction_inference
_predict_neural_original = prediction_inference._predict_neural
def _predict_neural_with_oom_retry(*args, batch_size=4, **kwargs):
    current_batch = min(int(batch_size), 4)
    while True:
        try:
            return _predict_neural_original(*args, batch_size=current_batch, **kwargs)
        except RuntimeError as exc:
            if 'out of memory' not in str(exc).lower() or current_batch <= 1:
                raise
            current_batch = max(1, current_batch // 2)
            print(f'CUDA out of memory; retrying {args[0]} with batch_size={current_batch}')
            torch.cuda.empty_cache()
prediction_inference._predict_neural = _predict_neural_with_oom_retry
print({'experiment_id': selection['experiment_id'], 'strategy': selection['selected_strategy'],
       'members': len(selection['required_candidates']), 'validation_accuracy': selection['validation_accuracy'],
       'device': 'cuda', 'test': str(TEST_CSV)})

In [ ]:
import pandas as pd

test = pd.read_csv(TEST_CSV, dtype={'id': 'string'})
assert {'id', 'text'}.issubset(test.columns), test.columns.tolist()
assert test['id'].notna().all() and not test['id'].duplicated().any()
test['text'] = test['text'].fillna('').astype(str)

predicted, classes = predict_prediction_bundle(BUNDLE_DIR, test['text'].tolist(), base_model_overrides=BASE_OVERRIDES)
submission = pd.DataFrame({'id': test['id'], 'model': predicted})
assert submission.columns.tolist() == ['id', 'model']
assert len(submission) == len(test)
assert submission['id'].astype(str).tolist() == test['id'].astype(str).tolist()
assert submission['model'].notna().all() and submission['model'].isin(classes).all()
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
submission.to_parquet(OUTPUT, index=False)
print({'rows': len(submission), 'output': str(OUTPUT), 'experiment_id': EXPECTED_EXPERIMENT_ID,
       'strategy': selection['selected_strategy'], 'validation_accuracy': EXPECTED_VALIDATION_ACCURACY})
display(submission.head())